# Week 20 - MLOps CI/CD and Monitoring (Data Engineer Variant)

Last week you used MLflow and the Model Registry to ship a fraud-detection
model behind the endpoint `fraud-classifier-endpoint`. This week we shift our
attention to what data engineers actually own in production: the PIPELINE
that feeds that model.

If the input distribution silently changes, the model still answers, just
wrongly. The data engineer is the first line of defense.

## What you will build today

1. A Delta Change Data Feed based freshness monitor that watches new rows
   arriving in `bread_academy.course_data.fraud_transactions` and computes
   how the new batch differs from a 30-day rolling baseline.
2. A PySpark data quality gate with four production-grade assertions, run
   before any downstream consumer touches the data.
3. A Delta audit log table at
   `bread_academy.student_work.pipeline_audit_log` so every pipeline run
   leaves a queryable trace you can join with MLflow runs later.
4. An automated retraining trigger that calls the Databricks Workflows
   REST API to start the ML engineer's retraining job when your data
   quality signals fire.

## Learning objectives

- Enable and read Delta Change Data Feed for pipeline observability.
- Express data quality checks as PySpark assertions you can run in CI.
- Write structured audit events to a Delta table.
- Trigger Databricks Workflows programmatically via REST.

## Environment Setup

**Platform**: Azure Databricks, plain Databricks Runtime 15.4 LTS (Python 3.10).

**Cluster**: A shared course cluster is provisioned for you. Confirm the
runtime in the cluster UI before attaching this notebook.

The next code cell uses `%pip install` (a Databricks magic) to install the
libraries the cluster does not have or whose bundled versions are too old.
After it finishes, `dbutils.library.restartPython()` restarts the Python
kernel for you - then re-run from the top.

**Libraries installed by the `%pip` cell**:

- `numpy<2`, `pandas<2` - pinned first so no transitive dependency bumps
  numpy to 2.x (the runtime pyarrow is compiled against numpy 1.x and a
  bump crashes the kernel)
- `boto3>=1.36` - the runtime-bundled boto3 predates Bedrock, so we install a current one
- `mlflow-skinny>=2.13,<3` - the course cluster is the PLAIN runtime, not
  the ML runtime, so mlflow does not ship with it; mlflow-skinny is the
  lightweight tracking client and does not pull numpy/pandas
- `requests>=2.31` - for the Databricks Workflows REST API call

`pyspark` ships with the runtime and is used FROM the cluster (not pip-installed).

**Secrets**: per-student AWS keys (`aws-access-key-id`,
`aws-secret-access-key`) come from this student's `aws-course-creds-NN`
scope (NN derived from your Databricks username); class-wide values
(`aws-region`, `databricks-token`, `retrain-job-id`) come from the
`aws-course-shared` scope. The course operators have already populated
both scopes.

**Unity Catalog permissions** (already granted to your group):
- USE CATALOG on `bread_academy`
- USE SCHEMA, SELECT on `bread_academy.course_data`
- USE SCHEMA, CREATE TABLE, MODIFY on `bread_academy.student_work`

In [ ]:
# Install libraries the cluster does not have or whose bundled versions are
# too old. Safe to re-run. restartPython() reloads the kernel afterwards.
# The course cluster is PLAIN DBR 15.4 LTS (not the ML runtime), so mlflow
# does NOT ship with it. We install mlflow-skinny: the lightweight tracking
# client, which does NOT pull numpy/pandas/scipy and so cannot break the
# kernel. numpy<2 / pandas<2 are pinned first so no transitive dependency
# bumps numpy to 2.x (DBR 15.4's pyarrow is compiled against numpy 1.x).
# pyspark IS cluster-provided.
%pip install --quiet "numpy<2" "pandas<2" "boto3>=1.36" "mlflow-skinny>=2.13,<3" "requests>=2.31"
dbutils.library.restartPython()

In [ ]:
from importlib.metadata import version
import os, json, time, uuid
from datetime import datetime, timedelta, timezone
import requests
import boto3
import mlflow
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, BooleanType, TimestampType

# mlflow is provided by the mlflow-skinny distribution on this cluster, so
# query the version under that distribution name.
print("mlflow-skinny:", version("mlflow-skinny"))
print("boto3:", version("boto3"))
print("requests:", version("requests"))

In [ ]:
# Derive this student's per-user secret scope from the Databricks identity.
# Per-student AWS keys live in aws-course-creds-NN; class-wide config lives
# in aws-course-shared.
_user = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().userName().get()
)
_num = _user.split("@")[0].split("-")[1] if _user.startswith("student-") else "01"
creds_scope = f"aws-course-creds-{_num}"

# AWS creds for any boto3 calls. Long-lived IAM-user keys (no session token).
os.environ["AWS_ACCESS_KEY_ID"]     = dbutils.secrets.get(scope=creds_scope, key="aws-access-key-id")
os.environ["AWS_SECRET_ACCESS_KEY"] = dbutils.secrets.get(scope=creds_scope, key="aws-secret-access-key")
os.environ["AWS_DEFAULT_REGION"]    = dbutils.secrets.get(scope="aws-course-shared", key="aws-region")

# Databricks token for the Workflows REST API call later (class-wide).
DATABRICKS_TOKEN = dbutils.secrets.get(scope="aws-course-shared", key="databricks-token")

# Workspace URL (returned without scheme).
DATABRICKS_HOST = "https://" + spark.conf.get("spark.databricks.workspaceUrl")
print("Workspace:", DATABRICKS_HOST)

# Names we will reuse throughout.
SOURCE_TABLE = "bread_academy.course_data.fraud_transactions"
AUDIT_TABLE = "bread_academy.student_work.pipeline_audit_log"
mlflow.set_tracking_uri("databricks")  # Databricks-native MLflow
MLFLOW_EXP = "/Shared/bread_academy/week20_pipeline_health"
ENDPOINT_NAME = "fraud-classifier-endpoint"  # consumer downstream, not modified here

In [ ]:
# Probe 1: source table is reachable
src_count = spark.sql(f"SELECT COUNT(*) AS c FROM {SOURCE_TABLE}").collect()[0]["c"]
print(f"Source table OK: {src_count:,} rows.")

# Probe 2: student_work schema is writable
try:
    spark.sql(
        "CREATE TABLE IF NOT EXISTS bread_academy.student_work._probe_w20 "
        "(id INT) USING DELTA"
    )
    spark.sql("DROP TABLE bread_academy.student_work._probe_w20")
    print("student_work schema: WRITE OK.")
except Exception as e:
    print("Ask your instructor to grant CREATE TABLE on bread_academy.student_work.")
    raise

# Probe 3: MLflow experiment is reachable
mlflow.set_experiment(MLFLOW_EXP)
print(f"MLflow experiment OK: {MLFLOW_EXP}")

## What Are We Building Today?

Imagine you are on call for the Bread Financial fraud platform. At 3am the
on-call ML engineer pages you: "the model is flagging twice as many txns as
yesterday, did something change upstream?" Without pipeline observability
your only answer is to start querying tables. With it, you already know:
the source table received 1.4M new rows in the last 6 hours (normal: 800k),
the merchant_category mix shifted hard toward category 42 (a new partner
went live), and the PSI score on `amount` is 0.31 (way above the 0.2 alert
threshold).

You did not write the model. You did not retrain it. But you SAVED the
model owner from a bad night because the pipeline told its own story.

That is what the next four topics build, end to end.

## Topic 1 - Delta Change Data Feed as a freshness monitor

### Why CDF over a periodic `SELECT COUNT(*)`?

A periodic count tells you how many rows exist at a point in time. CDF tells
you exactly which rows ARRIVED, in what order, with what values. You can
compare yesterday's slice against today's slice without touching historical
data, and you can do it incrementally (cheap) rather than full-scanning the
table (expensive).

### Enabling CDF is an instructor operation

CDF must be enabled before any change is captured - past versions of the
table are NOT retroactively recorded. Enabling it is an `ALTER TABLE`:

```python
spark.sql(
    "ALTER TABLE bread_academy.course_data.fraud_transactions "
    "SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
)
```

`ALTER TABLE ... SET TBLPROPERTIES` is a table-ownership operation in Unity
Catalog. The source table is shared across the whole class, so it is owned
by the instructor, not by students. Your instructor enables CDF once, before
class, in `instructor_setup_databricks.ipynb`. The next cell VERIFIES that
CDF is on - a read-only check that needs only `SELECT`.

### Reading the feed

```python
# startingVersion must be at or after the version where CDF was enabled.
# Reading from a version before enablement throws an error, not empty data.
df_changes = (
    spark.read.format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", cdf_start_version)  # computed, never hardcoded
    .table("bread_academy.course_data.fraud_transactions")
)
```

The result has the original columns plus `_change_type`, `_commit_version`,
and `_commit_timestamp`. We will use those three columns to slice the
"last hour" of arrivals.

In [ ]:
# Change Data Feed is enabled on the source table by your instructor
# (instructor_setup_databricks.ipynb does it once, before class). Enabling
# CDF is an ALTER TABLE - a table-ownership operation - so students do NOT
# run it. We VERIFY it here instead. This is read-only: it needs only
# SELECT, which you already have on the source table.
props = {
    r["key"]: r["value"]
    for r in spark.sql(f"SHOW TBLPROPERTIES {SOURCE_TABLE}").collect()
}
cdf_on = props.get("delta.enableChangeDataFeed", "false").lower() == "true"
if not cdf_on:
    raise RuntimeError(
        f"Change Data Feed is NOT enabled on {SOURCE_TABLE}. "
        "Ask your instructor to run instructor_setup_databricks.ipynb "
        "(it enables CDF on the source table)."
    )
print("Change Data Feed: ENABLED on", SOURCE_TABLE)

hist = spark.sql(f"DESCRIBE HISTORY {SOURCE_TABLE}")
latest_version = hist.agg(F.max("version").alias("v")).collect()[0]["v"]
print("Latest version:", latest_version)

# Find the version at which CDF was enabled. The change feed cannot be read
# from before this version, so we never start below it.
enable_rows = (
    hist.filter(F.col("operation") == "SET TBLPROPERTIES")
    .agg(F.min("version").alias("v"))
    .collect()
)
cdf_enable_version = enable_rows[0]["v"] if enable_rows and enable_rows[0]["v"] is not None else 0

# Safe starting version: a few versions back, but never before enablement.
start_v = max(latest_version - 5, cdf_enable_version)
print("CDF enabled at version:", cdf_enable_version, "| reading feed from:", start_v)

df_changes = (
    spark.read.format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", start_v)
    .table(SOURCE_TABLE)
)

inserts_preview = df_changes.filter(F.col("_change_type") == "insert")
n_inserts = inserts_preview.count()
if n_inserts == 0:
    print(
        "WARNING: the change feed has 0 post-enablement inserts. "
        "Ask your instructor to append a fresh batch of rows so the "
        "freshness and PSI labs have data to work with."
    )
display(
    inserts_preview
    .select("_commit_version", "_commit_timestamp", "amount", "is_fraud")
    .orderBy(F.desc("_commit_timestamp"))
    .limit(20)
)

## Lab 1 - Build a freshness monitor (10 minutes)

Build a function `freshness_snapshot(start_version)` that returns a single
Python dict with these keys:

- `rows_arrived` - count of inserts since `start_version`.
- `latest_commit_ts` - max `_commit_timestamp` seen (Python datetime).
- `fraud_rate_recent` - fraction of recent arrivals where `is_fraud == 1`.
- `fraud_rate_baseline` - overall fraud rate of the table.
- `fraud_rate_delta` - `fraud_rate_recent - fraud_rate_baseline`.

Then log all five values to MLflow under experiment `MLFLOW_EXP` as a
single run named `freshness-<timestamp>`.

You can reuse `start_v` from the demo cell as your starting version.

In [ ]:
# SOLUTION: Lab 1 - Freshness monitor
def freshness_snapshot(start_version: int) -> dict:
    inserts = (
        spark.read.format("delta")
        .option("readChangeFeed", "true")
        .option("startingVersion", start_version)
        .table(SOURCE_TABLE)
        .filter(F.col("_change_type") == "insert")
    )
    if inserts.count() == 0:
        # CDF feed empty (no post-enablement inserts). Fall back to a
        # time-travel slice: the rows added between the two latest versions.
        print("Change feed empty; falling back to a time-travel slice.")
        cur = spark.read.format("delta").option("versionAsOf", latest_version).table(SOURCE_TABLE)
        prev_v = max(latest_version - 1, 0)
        prev = spark.read.format("delta").option("versionAsOf", prev_v).table(SOURCE_TABLE)
        inserts = cur.join(prev, on="transaction_id", how="left_anti")
        ts_col = F.current_timestamp()
    else:
        ts_col = F.col("_commit_timestamp")
    agg = inserts.agg(
        F.count("*").alias("rows_arrived"),
        F.max(ts_col).alias("latest_commit_ts"),
        F.avg(F.col("is_fraud").cast("double")).alias("fraud_rate_recent"),
    ).collect()[0]
    baseline = spark.table(SOURCE_TABLE).agg(
        F.avg(F.col("is_fraud").cast("double")).alias("b")
    ).collect()[0]["b"]
    rows_arrived = int(agg["rows_arrived"] or 0)
    fraud_recent = float(agg["fraud_rate_recent"] or 0.0)
    fraud_baseline = float(baseline or 0.0)
    return {
        "rows_arrived": rows_arrived,
        "latest_commit_ts": agg["latest_commit_ts"],
        "fraud_rate_recent": fraud_recent,
        "fraud_rate_baseline": fraud_baseline,
        "fraud_rate_delta": fraud_recent - fraud_baseline,
    }

# Run it and log
mlflow.set_experiment(MLFLOW_EXP)
with mlflow.start_run(run_name=f"freshness-{int(time.time())}"):
    snap = freshness_snapshot(start_v)
    mlflow.log_metric("rows_arrived", snap["rows_arrived"])
    mlflow.log_metric("fraud_rate_recent", snap["fraud_rate_recent"])
    mlflow.log_metric("fraud_rate_baseline", snap["fraud_rate_baseline"])
    mlflow.log_metric("fraud_rate_delta", snap["fraud_rate_delta"])
    if snap["latest_commit_ts"] is not None:
        mlflow.set_tag("latest_commit_ts", str(snap["latest_commit_ts"]))

print(snap)

## Think About It

You enabled CDF on the source table. A teammate asks: "Can I get yesterday's
changes too? CDF should give me history, right?" What do you tell them?

(Hint: CDF captures changes from the moment it is enabled. Anything before
that is invisible to the feed. For history you would have to reconstruct
from `DESCRIBE HISTORY` and time travel queries, which is not the same
thing.)

## Topic 2 - A data quality gate before the model sees the data

The freshness monitor tells you WHAT changed. A data quality gate decides
whether the change is acceptable. If it is not, you HALT the pipeline so
the downstream consumer (the fraud model) never sees the bad batch.

We will use four checks that cover 80 percent of real production failures:

1. No nulls in `is_fraud` (the label column must always be present).
2. `amount > 0` on every row (no refunds, no test rows).
3. `partition_date` within the last 90 days (no historical replays).
4. Fraud rate between 1 percent and 20 percent (class balance sanity).

We implement them as plain PySpark assertions because (a) they are easy to
read, (b) they run on the same engine as the data, and (c) they do not add
a fragile dependency on Great Expectations for a beginner audience.

In [ ]:
def assert_no_null_labels(df, col_name="is_fraud"):
    n_null = df.filter(F.col(col_name).isNull()).count()
    if n_null > 0:
        raise AssertionError(f"DQ FAIL: {n_null} null values in '{col_name}'")
    return {"check": "no_null_labels", "passed": True, "n_null": 0}

# Demo on the current source table
result = assert_no_null_labels(spark.table(SOURCE_TABLE))
print(result)

## Lab 2 - Implement the remaining 3 checks (10 minutes)

Following the pattern in the demo, implement three functions. Each must
return a dict with at least `check`, `passed`, and one diagnostic field
(e.g., `n_bad`, `rate`, `min_date`). Each must raise `AssertionError` with
a clear message on failure:

1. `assert_positive_amount(df)` - all rows must have `amount > 0`.
2. `assert_recent_dates(df, days=90)` - all `partition_date` values within
   the last `days` days.
3. `assert_fraud_rate_in_band(df, lo=0.01, hi=0.20)` - overall fraud rate in
   `[lo, hi]`.

Then wrap all four in a `run_quality_gate(df)` function that runs them
sequentially, collects results in a list, and returns `(all_passed, results)`.
The gate must NOT short-circuit on the first failure - we want to see every
failing check in one shot.

In [ ]:
# SOLUTION: Lab 2 - Three more DQ checks + gate runner
def assert_positive_amount(df):
    n_bad = df.filter(F.col("amount") <= 0).count()
    passed = (n_bad == 0)
    if not passed:
        raise AssertionError(f"DQ FAIL: {n_bad} rows with amount <= 0")
    return {"check": "positive_amount", "passed": True, "n_bad": 0}

def assert_recent_dates(df, days=90):
    # partition_date is a DateType column; compare against a date literal.
    cutoff = (datetime.now(timezone.utc) - timedelta(days=days)).date()
    n_bad = df.filter(F.col("partition_date") < F.lit(cutoff)).count()
    passed = (n_bad == 0)
    if not passed:
        raise AssertionError(f"DQ FAIL: {n_bad} rows older than {days} days")
    return {"check": "recent_dates", "passed": True, "n_bad": 0}

def assert_fraud_rate_in_band(df, lo=0.01, hi=0.20):
    rate = df.agg(F.avg(F.col("is_fraud").cast("double")).alias("r")).collect()[0]["r"] or 0.0
    passed = (lo <= rate <= hi)
    if not passed:
        raise AssertionError(f"DQ FAIL: fraud rate {rate:.4f} outside [{lo}, {hi}]")
    return {"check": "fraud_rate_band", "passed": True, "rate": round(rate, 4)}

def run_quality_gate(df):
    results = []
    for fn in (assert_no_null_labels, assert_positive_amount, assert_recent_dates, assert_fraud_rate_in_band):
        try:
            results.append(fn(df))
        except AssertionError as e:
            results.append({"check": fn.__name__, "passed": False, "error": str(e)})
    return all(r.get("passed") for r in results), results

all_passed, gate_results = run_quality_gate(spark.table(SOURCE_TABLE))
print("ALL PASSED:", all_passed)
for r in gate_results:
    print(r)

## Topic 3 - Structured audit log

Now we have two signals (freshness snapshot, quality gate) and no place to
put them. MLflow is fine for ad-hoc metric logging but it is awkward to JOIN
against other tables for analytics. So we add a second store: a Delta table
in `bread_academy.student_work.pipeline_audit_log` that mirrors every run.

### Schema we will write

| Column | Type | Meaning |
|--------|------|---------|
| run_id | STRING | UUID, one per pipeline invocation |
| run_timestamp | TIMESTAMP | When the run completed |
| rows_processed | LONG | Total rows seen this run |
| fraud_rate | DOUBLE | Observed fraud rate this run |
| psi_score | DOUBLE | PSI of `amount` against the 30-day baseline |
| drift_detected | BOOLEAN | True if any DQ check failed or PSI > 0.2 |
| action_taken | STRING | One of: `none`, `alerted`, `retrain_triggered` |

The first write creates the table; subsequent writes append. Because we
own MODIFY on `student_work`, this is fully self-service.

In [ ]:
audit_schema = StructType([
    StructField("run_id", StringType()),
    StructField("run_timestamp", TimestampType()),
    StructField("rows_processed", LongType()),
    StructField("fraud_rate", DoubleType()),
    StructField("psi_score", DoubleType()),
    StructField("drift_detected", BooleanType()),
    StructField("action_taken", StringType()),
])

demo_row = [(
    str(uuid.uuid4()),
    datetime.now(timezone.utc),
    int(snap["rows_arrived"]),
    float(snap["fraud_rate_recent"]),
    0.05,        # placeholder PSI, we compute the real one in Lab 3
    False,
    "none",
)]

(
    spark.createDataFrame(demo_row, schema=audit_schema)
    .write.mode("append")
    .saveAsTable(AUDIT_TABLE)
)

display(spark.table(AUDIT_TABLE).orderBy(F.desc("run_timestamp")).limit(5))

## Lab 3 - Compute PSI on `amount` and write a real audit row (10 minutes)

PSI (Population Stability Index) on a continuous column is computed by:

1. Bucketing the baseline distribution into N (use 10) quantile bins.
2. Computing the proportion of rows in each bin for baseline and recent.
3. PSI = sum_i (recent_i - baseline_i) * ln(recent_i / baseline_i),
   with epsilon = 1e-6 to avoid divide-by-zero.

Implement:

- `psi_amount(df_baseline, df_recent, n_bins=10) -> float`
- `write_audit_row(snap, psi, drift_detected, action_taken)` that appends
  one row to `AUDIT_TABLE` using `audit_schema`.

Then call them. Use the full source table as the baseline and the CDF
inserts slice as the recent population.

In [ ]:
# SOLUTION: Lab 3 - PSI on amount + audit row writer
import math

def psi_amount(df_baseline, df_recent, n_bins=10):
    qs = [i / n_bins for i in range(1, n_bins)]
    edges = df_baseline.approxQuantile("amount", qs, 0.001)
    edges = [float("-inf")] + edges + [float("inf")]

    def bucket_props(df):
        cases = F.when(F.col("amount") < edges[1], 0)
        for i in range(1, len(edges) - 1):
            cases = cases.when(F.col("amount") < edges[i + 1], i)
        df2 = df.withColumn("_b", cases)
        total = df2.count() or 1
        props = (
            df2.groupBy("_b").count()
            .withColumn("p", F.col("count") / F.lit(total))
            .select("_b", "p").collect()
        )
        return {r["_b"]: r["p"] for r in props}

    b = bucket_props(df_baseline)
    r = bucket_props(df_recent)
    eps = 1e-6
    psi = 0.0
    for i in range(n_bins):
        bp = b.get(i, 0.0) + eps
        rp = r.get(i, 0.0) + eps
        psi += (rp - bp) * math.log(rp / bp)
    return float(psi)

def write_audit_row(snap, psi, drift_detected, action_taken):
    row = [(
        str(uuid.uuid4()),
        datetime.now(timezone.utc),
        int(snap["rows_arrived"]),
        float(snap["fraud_rate_recent"]),
        float(psi),
        bool(drift_detected),
        str(action_taken),
    )]
    (
        spark.createDataFrame(row, schema=audit_schema)
        .write.mode("append")
        .saveAsTable(AUDIT_TABLE)
    )

def cdf_recent_population(start_version):
    """Recent-arrivals population for PSI. Uses the CDF insert slice, with a
    time-travel fallback when the change feed has no post-enablement rows."""
    recent = (
        spark.read.format("delta")
        .option("readChangeFeed", "true")
        .option("startingVersion", start_version)
        .table(SOURCE_TABLE)
        .filter(F.col("_change_type") == "insert")
        .select(*spark.table(SOURCE_TABLE).columns)
    )
    if recent.count() == 0:
        print("Change feed empty; PSI recent population from time-travel slice.")
        cur = spark.read.format("delta").option("versionAsOf", latest_version).table(SOURCE_TABLE)
        prev_v = max(latest_version - 1, 0)
        prev = spark.read.format("delta").option("versionAsOf", prev_v).table(SOURCE_TABLE)
        recent = cur.join(prev, on="transaction_id", how="left_anti")
        if recent.count() == 0:
            # No new rows at all; use a random sample so PSI is still defined.
            recent = spark.table(SOURCE_TABLE).sample(fraction=0.1, seed=20)
    return recent

baseline_df = spark.table(SOURCE_TABLE)
recent_df = cdf_recent_population(start_v)
psi_value = psi_amount(baseline_df, recent_df)

drift = (not all_passed) or (psi_value > 0.2)
write_audit_row(snap, psi_value, drift, "none")
display(spark.table(AUDIT_TABLE).orderBy(F.desc("run_timestamp")).limit(3))

## Think About It

You now write to BOTH MLflow and a Delta audit table. That feels redundant.
When is each one the right home?

(Hint: MLflow is shaped around RUNS with parameters/metrics/artifacts and
shines for experiment comparison. A Delta audit table is shaped around
ROWS you can JOIN against transactions, customer data, or alert history.
Operationally you usually need both.)

## Topic 4 - Triggering retraining via the Databricks Workflows REST API

Detecting drift without acting on it is theater. The data engineer's job is
to hand off cleanly: when our signals fire, we start the ML engineer's
retraining job programmatically.

### The call

`POST {DATABRICKS_HOST}/api/2.1/jobs/run-now`

Headers: `Authorization: Bearer {DATABRICKS_TOKEN}`,
         `Content-Type: application/json`.

Body:
```json
{
  "job_id": 123456789,
  "notebook_params": {
    "trigger_reason": "drift_detected",
    "psi_score": "0.31",
    "audit_run_id": "..."
  }
}
```

Response (on success): `{ "run_id": ..., "number_in_job": ... }`. We
record the `run_id` in our audit table so the ML engineer can follow the
chain back from a retraining run to the data signals that caused it.

For class, the instructor has pre-created a tiny "stub" Databricks Job
that just prints its parameters; the job id is in the `aws-course-shared`
scope as `retrain-job-id`.

In [ ]:
RETRAIN_JOB_ID = int(dbutils.secrets.get(scope="aws-course-shared", key="retrain-job-id"))

def trigger_retraining(job_id, payload_params):
    """Start a Databricks job by id. Tries notebook_params first; if the job
    was defined with job-level parameters, retries with job_parameters."""
    url = f"{DATABRICKS_HOST}/api/2.1/jobs/run-now"
    headers = {
        "Authorization": f"Bearer {DATABRICKS_TOKEN}",
        "Content-Type": "application/json",
    }
    for param_key in ("notebook_params", "job_parameters"):
        body = {"job_id": job_id, param_key: payload_params}
        r = requests.post(url, headers=headers, data=json.dumps(body), timeout=20)
        if r.status_code == 400 and "parameter" in r.text.lower() and param_key == "notebook_params":
            # Job declares job-level parameters; retry with job_parameters.
            continue
        r.raise_for_status()
        return r.json()
    r.raise_for_status()
    return r.json()

resp = trigger_retraining(
    RETRAIN_JOB_ID,
    {"trigger_reason": "demo", "psi_score": "0.0", "audit_run_id": "demo"},
)
print(resp)
print("Triggered run_id:", resp.get("run_id"))

## Lab 4 - Glue everything (15 minutes)

Implement `pipeline_run()` that ties the four topics together:

1. Compute the freshness snapshot (reuse `freshness_snapshot`).
2. Run the quality gate (reuse `run_quality_gate`).
3. Compute PSI on `amount` (reuse `psi_amount`).
4. Decide an `action_taken`:
   - `none` if all checks passed and PSI <= 0.2.
   - `alerted` if any check failed OR PSI > 0.2 but PSI <= 0.4.
   - `retrain_triggered` if PSI > 0.4 OR more than one gate check failed.
5. If `action_taken == "retrain_triggered"`, call `trigger_retraining`
   with `trigger_reason="drift_detected"` and the PSI score, then record
   the returned `run_id` in the audit row's `action_taken` string as
   `f"retrain_triggered:{run_id}"`.
6. Write the audit row.

Return the final dict.

You can call your own `pipeline_run()` at the end of the cell to see it
behave end-to-end. Because the source table is mostly stable, you will
typically see `action_taken="none"`. That is correct; the alerting and
retraining branches are exercised by the homework data.

In [ ]:
# SOLUTION: Lab 4 - End-to-end pipeline_run
def pipeline_run() -> dict:
    snap_local = freshness_snapshot(start_v)
    passed_local, results_local = run_quality_gate(spark.table(SOURCE_TABLE))
    baseline_local = spark.table(SOURCE_TABLE)
    recent_local = cdf_recent_population(start_v)
    psi_local = psi_amount(baseline_local, recent_local)
    n_failed = sum(1 for r in results_local if not r.get("passed"))

    if psi_local > 0.4 or n_failed > 1:
        action = "retrain_triggered"
    elif (not passed_local) or psi_local > 0.2:
        action = "alerted"
    else:
        action = "none"

    drift_local = (not passed_local) or psi_local > 0.2

    if action == "retrain_triggered":
        resp_local = trigger_retraining(
            RETRAIN_JOB_ID,
            {
                "trigger_reason": "drift_detected",
                "psi_score": str(round(psi_local, 4)),
                "audit_run_id": "auto",
            },
        )
        action = f"retrain_triggered:{resp_local.get('run_id')}"

    write_audit_row(snap_local, psi_local, drift_local, action)
    return {
        "rows_arrived": snap_local["rows_arrived"],
        "fraud_rate_recent": snap_local["fraud_rate_recent"],
        "psi": psi_local,
        "drift_detected": drift_local,
        "action_taken": action,
        "gate_results": results_local,
    }

final = pipeline_run()
print(final)
display(spark.table(AUDIT_TABLE).orderBy(F.desc("run_timestamp")).limit(5))

## Recap

You shipped four pipeline-side MLOps building blocks today:

- **Freshness monitor** with Delta CDF, logged to MLflow.
- **Quality gate** with four PySpark assertions, halts on bad data.
- **Audit log** in Delta at `bread_academy.student_work.pipeline_audit_log`.
- **Auto retraining trigger** via Databricks Workflows REST API.

Notice what you did NOT do: you did not retrain the model, you did not
touch SageMaker, you did not edit the model code. As the data engineer
you OWNED the pipeline observability and HANDED OFF to the model owner.
That separation of duties is what makes MLOps survive contact with
production.

## Homework (async, ~45 min)

1. Add a fifth quality check: schema check. Read the schema of the source
   table once and store it as a tuple of (name, dtype) pairs. On every
   run, assert that the current schema is identical. Add it to
   `run_quality_gate`.
2. Compute PSI on a categorical column (`merchant_category`). PSI on
   categorical is the same formula but bins are category values, not
   quantiles. Add this as a second drift signal alongside `psi_amount`.
3. Build a small SQL query against `pipeline_audit_log` that returns the
   last 7 days of runs, with one row per day showing `n_runs`,
   `n_drift_detected`, `n_retrain_triggered`, `avg_psi`. This is the kind
   of pane an on-call data engineer wants pinned on a dashboard.

## Further reading

- Delta Change Data Feed: docs.databricks.com/aws/en/delta/delta-change-data-feed
- Jobs API run-now: docs.databricks.com/api/workspace/jobs/runnow
- MLflow experiments in jobs: docs.databricks.com/aws/en/mlflow/experiments
- Population Stability Index reference: see the Week 19 reading list.